In [26]:
import pandas as pd
import os
import numpy as np
import json

In [2]:
Dir_dataset = '/work/desai-lab/ABC_dataset/'
Dir_raw = '/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/'
# Dir_github = '/work/xy6/Project/ABC/data/'
Dir_github = '/work/svs-lab/xuanyang/Project/NSF_valence/github'
Dir_preprocess = os.path.join(Dir_dataset,'derivatives','fmriprep')
repo = 'ABC_valence'

In [3]:
# Demographic info 
df_behav = pd.read_csv(os.path.join(Dir_dataset,'codes','ABCStudyBehavioral_DATA_2024-08-13_0912.csv'))
# df_behav.loc[~df_behav['subject_id'].str.contains('|'.join(['test','Test','Prac','TEST','EXAMPLE']))]
# df_behav.loc[df_behav['subject_id'].str.contains('ABC')]
# df_behav = df_behav.loc[~df_behav['confirm_abcid'].isna()]

df_behav['subid'] = df_behav['subject_id'].values
df_behav = df_behav.loc[df_behav['subid'].str.contains('ABC')]
df_behav['subid'] = df_behav['subid'].str.replace('*','')
df_behav.loc[df_behav['subid'].str.contains('1068'),'subid'] = 'ABC1068'
df_behav = df_behav.loc[df_behav['subid'].str.len()==7]
df_behav['subid'] = df_behav['subid'].str.replace('ABC','').astype(int)
df_master_demo = df_behav[['subid','subject_id','sex','age_dayofday1testing','handedness','first_language','other_language']].copy()
df_master_demo = df_master_demo.loc[~df_master_demo['age_dayofday1testing'].isna()]
df_master_demo

,subid,subject_id,sex,age_dayofday1testing,handedness,first_language,other_language
0,1001,ABC1001,2.0,65.0,2.0,1.0,NaN
1,1002,ABC1002,1.0,61.0,2.0,1.0,NaN
2,1003,ABC1003,2.0,75.0,2.0,1.0,NaN
3,1004,ABC1004,2.0,75.0,3.0,1.0,NaN
4,1005,ABC1005,2.0,65.0,2.0,1.0,NaN
...,...,...,...,...,...,...,...
285,1349,ABC1349,2.0,21.0,2.0,1.0,NaN
286,1350,ABC1350,1.0,61.0,2.0,1.0,NaN
287,1351,ABC1351,2.0,36.0,2.0,1.0,NaN
288,1352,ABC1352,1.0,60.0,1.0,1.0,NaN


In [4]:
Dir_raw

'/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/'

In [5]:
# MRI raw data
List_files1 = os.listdir(Dir_raw)
List_files2 = os.listdir(os.path.join(Dir_raw,'ABC_2nd_batch'))
df_mri1 = pd.DataFrame({'raw':List_files1,'path':Dir_raw})
df_mri2 = pd.DataFrame({'raw':List_files2,'path':os.path.join(Dir_raw,'ABC_2nd_batch')})
df_mri = pd.concat([df_mri1,df_mri2])
# df_mri = df_mri.loc[df_mri['raw'].str.len()==7]
df_mri = df_mri.loc[df_mri['raw'].str.contains('ABC')]
df_mri = df_mri.loc[~df_mri['raw'].str.contains('_')]
df_mri['sesid'] = 1
df_mri.loc[df_mri['raw'].str.len()==8,'sesid'] = df_mri.loc[df_mri['raw'].str.len()==8,'raw'].str[-1].astype(int)
df_mri['subid'] = df_mri['raw'].str[0:7].str.replace('ABC','').astype(int)
df_mri.sort_values('subid',inplace=True)
df_mri
# df_mri.loc[df_mri['raw'].str.len()==8,'sesID'] = 

,raw,path,sesid,subid
135,ABC1001,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/,1,1001
157,ABC1002,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/,1,1002
104,ABC10022,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/,2,1002
129,ABC10032,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/,2,1003
107,ABC1003,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/,1,1003
...,...,...,...,...
37,ABC1321,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/...,1,1321
43,ABC1322,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/...,1,1322
28,ABC1323,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/...,1,1323
25,ABC1326,/work/desai-lab/ABC_raw/__ABC_MRI_RAW-restart/...,1,1326


In [6]:
meta_master = {}
meta_master['mri_source'] = {}
meta_master['mri_raw'] = {}
meta_master['mri_preproc'] = {}
tasks = ['Fam','Passage','rest']


for i,row in df_mri.drop_duplicates(['subid'], keep='first').iterrows():
    subid = row['subid']
    subID = f'sub-{subid}'
    
    

    # source data from ABC
    meta_master['mri_source'][subid] = {}
    path = os.path.join(row['path'],row['raw'],'ABC')
    meta_master['mri_source'][subid]['path'] = path
    meta_master['mri_source'][subid]['files'] = os.listdir(path)

    # raw data after organization, renaming
    path = os.path.join(Dir_dataset,'dataset',f'sub-{subid}')
    meta_master['mri_raw'][subid] = {}
    meta_master['mri_raw'][subid]['path'] = path
    visits = [x for x in os.listdir(path) if 'visit' in x]
    for visitID in visits:
        visit = visitID
        meta_master['mri_raw'][subid][visit] = {}
        # T1
        folder = os.path.join(path,visit,'anat')
        img = f"sub-{subid}_ses-{visit}_T1w.nii"
        if os.path.exists(os.path.join(folder,img)):
            meta_master['mri_raw'][subid][visit]['anat'] = img
        else:
            meta_master['mri_raw'][subid][visit]['anat'] = np.nan

        # fmri
        meta_master['mri_raw'][subid][visit]['func'] = {}
        folder = os.path.join(path,visit,'func')
        for task in tasks:
            img = f"sub-{subid}_ses-{visit}_task-{task}_bold.nii"
            if os.path.exists(os.path.join(folder,img)):
                meta_master['mri_raw'][subid][visit]['func'][task] = img
            else:
                meta_master['mri_raw'][subid][visit]['func'][task] = np.nan

    # preprocessed
    pipeline = 'fmriprep'
    path = os.path.join(Dir_dataset,'derivatives',pipeline,subID)
    if not os.path.exists(path):
        continue
    meta_master['mri_preproc'][subid] = {}
    meta_master['mri_preproc'][subid][pipeline] = {}
    meta_master['mri_preproc'][subid][pipeline]['path'] = path
    visits = [x for x in os.listdir(path) if 'visit' in x]
    
    for visitID in visits:
        visit = visitID.replace('ses-','')
        meta_master['mri_preproc'][subid][pipeline][visit] = {}
        folder = os.path.join(path,visitID,'anat')
        img = f"sub-{subid}_ses-{visit}_desc-preproc_T1w.nii.gz"
        if os.path.exists(os.path.join(folder,img)):
            meta_master['mri_preproc'][subid][pipeline][visit]['anat'] = os.path.join(folder,img)
        else:
            folder = os.path.join(path,'anat')
            img = f"sub-{subid}_desc-preproc_T1w.nii.gz"
            if os.path.exists(os.path.join(folder,img)):
                meta_master['mri_preproc'][subid][pipeline][visit]['anat'] = os.path.join(folder,img)
            else:
                meta_master['mri_preproc'][subid][pipeline][visit]['anat'] = np.nan
                print(subid,visit,'no preprocessed T1.')

        # fmri
        space = 'MNI152NLin2009cAsym'
        meta_master['mri_preproc'][subid][pipeline][visit]['func'] = {}
        meta_master['mri_preproc'][subid][pipeline][visit]['func'][space] = {}
        folder = os.path.join(path,visitID,'func')
        for task in tasks:
            
            img = f"sub-{subid}_ses-{visit}_task-{task}_space-{space}_res-2_desc-preproc_bold.nii.gz"
            if os.path.exists(os.path.join(folder,img)):
                meta_master['mri_preproc'][subid][pipeline][visit]['func'][space][task] = os.path.join(folder,img)
            else:
                meta_master['mri_preproc'][subid][pipeline][visit]['func'][space][task] = np.nan
                # print(subid,visit,f'no preprocessed {task}.')

# meta_master

    # sub-1051_ses-visit01_task-Fam_space-MNI152NLin2009cAsym_res-2_desc-preproc_bold
    

1035 visit01 no preprocessed T1.
1035 visit02 no preprocessed T1.


In [7]:
pipeline = 'fmriprep'
df_master_fmri = []
for subid in meta_master['mri_source']:
    if subid not in meta_master['mri_preproc']:
        visits = [x for x in meta_master['mri_raw'][subid] if 'visit' in x]
        for visit in visits:
            df_master_fmri.append(pd.DataFrame({'subid':subid,'visit':visit,'no_preproc':1},index=[0]))
        # print(subid)
        continue
    
    visits = [x for x in meta_master['mri_preproc'][subid][pipeline] if 'visit' in x]
    for visit in visits:
        df = pd.DataFrame(meta_master['mri_preproc'][subid][pipeline][visit]['func']).T
        df['subid'] = subid
        df['visit'] = visit
        df['anat'] = meta_master['mri_preproc'][subid][pipeline][visit]['anat']
        df_master_fmri.append(df)
    # df[]
df_master_fmri = pd.concat(df_master_fmri,ignore_index=True)
df_master_fmri['visit'] = df_master_fmri['visit'].str.replace('visit0','').astype(int)
df_master_fmri

,Fam,Passage,rest,subid,visit,anat,no_preproc
0,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1001,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1002,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
2,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1002,2,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
3,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1003,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
4,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1003,2,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
...,...,...,...,...,...,...,...
246,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1321,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
247,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1322,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
248,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,1323,1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
249,NaN,NaN,NaN,1326,1,NaN,1.0


In [28]:
file_name = "ABC_meta_scans.json"

Dir_output = os.path.join(Dir_github,repo,'scripts','fMRI','master')
file_path = os.path.join(Dir_output,file_name)
with open(file_path, 'w') as json_file:
    json.dump(meta_master, json_file, indent=4)  # Use `indent=4` for pretty formatting

# print(f"Nested dictionary has been exported to {file_name}")

In [25]:
meta_master['mri_preproc'][1001]['fmriprep']['visit01']['anat']

'/work/desai-lab/ABC_dataset/derivatives/fmriprep/sub-1001/ses-visit01/anat/sub-1001_ses-visit01_desc-preproc_T1w.nii.gz'

In [13]:
df_master_fmri.loc[df_master_fmri['visit']==1,'anat'].unique().shape[0]

216

In [8]:
df_epirme = pd.read_csv('/work/svs-lab/xuanyang/Project/NSF_valence/github/ABC_valence/scripts/Behavioral/check_eprime/df_master_Eprime.csv')
df_epirme.rename({'sessionid':'visit','subid_raw':'subid'},axis=1,inplace=True)
df1 = df_epirme.loc[df_epirme['task']=='familarity',['subid','visit','edat','txt']]
df1.columns = ['subid','visit','Fam_edat','Fam_txt']
df2 = df_epirme.loc[df_epirme['task']=='passage',['subid','visit','edat','txt']]
df2.columns = ['subid','visit','Passage_edat','Passage_txt']

# compile
df_master_epirme = df1.merge(df2,on=['subid','visit'],how='outer')
df_master_fmri['subid'] = df_master_fmri['subid'].astype(int)
df_master = df_master_epirme.merge(df_master_fmri,on=['subid','visit'],how='outer')
# df_master = df_master.merge(df_master_demo,on=['subid'],how='outer')

df_master_raw = df_master.copy()

df_master = df_master.loc[(df_master['subid']<2000)&(df_master['subid']>1000)]
df_master = df_master.loc[df_master['subid']<=1323]
# probematic
df_master_prob = df_master.loc[(df_master['subid']>2000)|(df_master['subid']<1000)]
df_master

,subid,visit,Fam_edat,Fam_txt,Passage_edat,Passage_txt,Fam,Passage,rest,anat,no_preproc
13,1001,1,NaN,NaN,NaN,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
14,1001,5,fMRI_fam-1001-5,fMRI_fam-1001-5,Passage_fMRI-1001-5,Passage_fMRI-1001-5,NaN,NaN,NaN,NaN,NaN
15,1002,1,NaN,NaN,NaN,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
16,1002,2,fMRI_fam-1002-2,fMRI_fam-1002-2,Passage_fMRI-1002-2,Passage_fMRI-1002-2,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
17,1002,5,fMRI_fam-1002-5,fMRI_fam-1002-5,Passage_fMRI-1002-5,Passage_fMRI-1002-5,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
291,1320,1,fMRI_fam-1320-1,fMRI_fam-1320-1,Passage_fMRI-1320-1,Passage_fMRI-1320-1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
292,1321,1,fMRI_fam-1321-1,fMRI_fam-1321-1,Passage_fMRI-1321-1,Passage_fMRI-1321-1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
293,1322,1,fMRI_fam-1322-1,NaN,Passage_fMRI-1322-1,Passage_fMRI-1322-1,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,/work/desai-lab/ABC_dataset/derivatives/fmripr...,NaN
294,1322,1322,NaN,fMRI_fam-1322-1322,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df_summary = []
for task in ['Passage','Fam']:
    List_vars = ['subid','visit','anat','no_preproc']+[x for x in df_master.columns if task in x]
    # df = df_master.loc[(~df_master[task].isna())&((df_master[f'{task}_edat'].isna())&(df_master[f'{task}_txt'].isna())),List_vars]
    df = df_master.loc[(~df_master[task].isna())&((df_master[f'{task}_edat'].isna())),List_vars]
    df['task'] = task
    df['notes'] = 'Have fmri image, but no eprime files.'
    df_summary.append(df[['subid','visit','task','notes','no_preproc']])

    df = df_master.loc[(df_master[task].isna())&((~df_master[f'{task}_edat'].isna())),List_vars]
    df = df.loc[df['no_preproc'].isna()]
    df['task'] = task
    df['notes'] = 'No fmri image, but have eprime files.'
    df_summary.append(df[['subid','visit','task','notes','no_preproc']])

df_summary = pd.concat(df_summary)
df_summary = df_summary[['subid','visit','task','notes']]

df_summary.to_csv(os.path.join(Dir_github,repo,'scripts','Behavioral','check_eprime','Summary_matchImageEprime.csv'),index=False)
df_summary

,subid,visit,task,notes
13,1001,1,Passage,"Have fmri image, but no eprime files."
15,1002,1,Passage,"Have fmri image, but no eprime files."
18,1003,1,Passage,"Have fmri image, but no eprime files."
20,1004,1,Passage,"Have fmri image, but no eprime files."
61,1030,2,Passage,"Have fmri image, but no eprime files."
...,...,...,...,...
210,1196,1,Fam,"No fmri image, but have eprime files."
211,1197,1,Fam,"No fmri image, but have eprime files."
230,1224,1,Fam,"No fmri image, but have eprime files."
231,1226,1,Fam,"No fmri image, but have eprime files."


In [ ]:
#Delete

In [70]:
print("by subid 1362")
print(f"T1 scans: {df_master['T1'].dropna().shape[0]}")

print(f"Complete Passage data: {df_master[['T1','Passage_fmri','Passage_eprime']].dropna().shape[0]}")
print(f"Complete Familarity data: {df_master[['T1','Fam_fmri','Fam_eprime']].dropna().shape[0]}")

by subid 1362
T1 scans: 238
Complete Passage data: 191
Complete Familarity data: 201


In [81]:
dfs_eprime = pd.read_csv(os.path.join(Dir_dataset,'codes','fmri_passage_1001-1362.csv'))
dfs_eprime[['Subject','name']].dropna().groupby('name').count().sort_values('Subject',ascending=False)

,Subject
name,
thai_male_low.wav,85
hindi_male_low.wav,85
thai_female_high.wav,68
hindi_female_high.wav,68
thai_female_low.wav,47
hindi_female_low.wav,47
hindi_male_high.wav,45
thai_male_high.wav,44
Jacob_FireSelected.wav,13


# Old

In [18]:
Dir_fmriprep = os.path.join(Dir_dataset,'derivatives','fmriprep')
for i,row in df_master.iterrows():
    subid = row['subid']
    visit = row['session']
    Dir_subj = os.path.join(Dir_fmriprep,f'sub-{subid}')

    if not os.path.exists(Dir_subj):
        print(f'sub-{subid} does not have mri data')
        continue
        
    # T1
    path = os.path.join(Dir_subj,'ses-visit01','anat',f"sub-{subid}_ses-visit01_space-MNI152NLin2009cAsym_res-2_desc-preproc_T1w.nii.gz")
    if os.path.exists(path):
        df_master.loc[i,'T1'] = path
    else:
        path = os.path.join(Dir_subj,'ses-visit01','anat',f"sub-{subid}_ses-visit01_space-MNI152NLin2009cAsym_res-2_desc-preproc_T1w.nii.gz")
    
        print(f"sub-{subid}: no T1")

# List_folders = [x for x in os.listdir(Dir_fmriprep) if '.html' not in x and 'sub-' in x]
df_master

sub-1002: no T1
sub-1003: no T1
sub-1004: no T1
sub-1005: no T1
sub-1009: no T1
sub-1010 does not have mri data
sub-1011 does not have mri data
sub-1013 does not have mri data
sub-1014: no T1
sub-1016: no T1
sub-1017 does not have mri data
sub-1019: no T1
sub-1022: no T1
sub-1023: no T1
sub-1025: no T1
sub-1029: no T1
sub-1030: no T1
sub-1031: no T1
sub-1033: no T1
sub-1034 does not have mri data
sub-1035: no T1
sub-1039: no T1
sub-1042: no T1
sub-1043 does not have mri data
sub-1050: no T1
sub-1051: no T1
sub-1053: no T1
sub-1062: no T1
sub-1063 does not have mri data
sub-1064: no T1
sub-1065: no T1
sub-1075 does not have mri data
sub-1110 does not have mri data
sub-1116 does not have mri data
sub-1117 does not have mri data
sub-1127 does not have mri data
sub-1158 does not have mri data
sub-1163 does not have mri data
sub-1168 does not have mri data
sub-1170 does not have mri data
sub-1179 does not have mri data
sub-1188 does not have mri data
sub-1189 does not have mri data
sub-1191

,subid,subject_id,confirm_abcid,sex,age_dayofday1testing,handedness,first_language,other_language,T1
0,1001,ABC1001,ABC1001,2.0,65.0,2.0,1.0,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...
1,1002,ABC1002,ABC1002,1.0,61.0,2.0,1.0,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...
2,1003,ABC1003,ABC1003,2.0,75.0,2.0,1.0,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...
3,1004,ABC1004,ABC1004,2.0,75.0,3.0,1.0,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...
4,1005,ABC1005,ABC1005,2.0,65.0,2.0,1.0,NaN,/work/desai-lab/ABC_dataset/derivatives/fmripr...
...,...,...,...,...,...,...,...,...,...
285,1349,ABC1349,ABC1349,2.0,21.0,2.0,1.0,NaN,False
286,1350,ABC1350,ABC1350,1.0,61.0,2.0,1.0,NaN,False
287,1351,ABC1351,ABC1351,2.0,36.0,2.0,1.0,NaN,False
288,1352,ABC1352,ABC1352,1.0,60.0,1.0,1.0,NaN,False


In [ ]:
dfs_eprime = pd.read_csv(os.path.join(Dir_dataset,'codes','fmri_passage_1001-1362.csv'))
# dfs_eprime
df_master_eprime = dfs_eprime.groupby(['Subject','Session']).count().reset_index()
df_master_eprime = df_master_eprime[['Subject','Session']]
df_master_eprime['Passage_eprime'] = 1
df_master_eprime.rename({'Subject':'subid','Session':'visit'},axis=1,inplace=True)
df_master_eprime = df_master_eprime.loc[df_master_eprime['subid']>1000]
df_master_eprime
# dfs_eprime['Clock.Information']

### Check Eprime

In [14]:
# fmri
List_folders = [x for x in os.listdir(Dir_preprocess) if '.html' not in x and 'sub' in x]
df_master_fmri = []
for subID in List_folders:
    List_visits = [x for x in os.listdir(os.path.join(Dir_preprocess,subID)) if 'visit' in x]
    if len(List_visits)>0:
        for visit in List_visits:
            List_files = os.listdir(os.path.join(Dir_preprocess,subID,visit,'func'))
            
            for task in ['rest','Passage','Fam']:
                List_imgs = [x for x in List_files if task in x and 'preproc_bold.nii.gz' in x]
                if len(List_imgs)>0:
                    df_master_fmri.append(pd.DataFrame({'subid':subID,'visit':visit,'task':task+'_fmri','fmri':1},index=[0]))
df_master_fmri = pd.concat(df_master_fmri,ignore_index=True)
df_master_fmri = df_master_fmri.pivot(index=['subid','visit'],columns='task',values='fmri').reset_index()

df_master_fmri['subid'] = df_master_fmri['subid'].str.replace('sub-','').astype(int)
df_master_fmri['visit'] = df_master_fmri['visit'].str.replace('ses-visit0','').astype(int)
df_master_fmri

task,subid,visit,Fam_fmri,Passage_fmri,rest_fmri
0,1001,1,1.0,1.0,1.0
1,1002,1,1.0,1.0,1.0
2,1002,2,1.0,1.0,1.0
3,1003,1,1.0,1.0,1.0
4,1003,2,1.0,1.0,1.0
...,...,...,...,...,...
233,1318,1,1.0,1.0,1.0
234,1320,1,1.0,1.0,1.0
235,1321,1,1.0,1.0,1.0
236,1322,1,1.0,1.0,1.0


In [17]:
dfs_eprime = pd.read_csv(os.path.join(Dir_dataset,'codes','fmri_passage_1001-1362.csv'))
# dfs_eprime
df_master_eprime = dfs_eprime.groupby(['Subject','Session']).count().reset_index()
df_master_eprime = df_master_eprime[['Subject','Session']]
df_master_eprime['Passage_eprime'] = 1
df_master_eprime.rename({'Subject':'subid','Session':'visit'},axis=1,inplace=True)
df_master_eprime = df_master_eprime.loc[df_master_eprime['subid']>1000]
df_master_eprime
# dfs_eprime['Clock.Information']

,subid,visit,Passage_eprime
3,1002,2,1
4,1002,5,1
5,1003,2,1
6,1005,3,1
7,1006,1,1
...,...,...,...
240,10335,1,1
241,10425,5,1
242,10445,1,1
243,10535,1,1


In [30]:
df_master_combine = df_master_fmri.merge(df_master_eprime,on=['subid','visit'],how='outer')
List_vars = ['subid','visit','Passage_fmri','Passage_eprime']
df_master_combine.loc[df_master_combine[List_vars].isna().any(axis=1),List_vars].head(60)

,subid,visit,Passage_fmri,Passage_eprime
0,1001,1,1.0,NaN
1,1002,1,1.0,NaN
3,1002,5,NaN,1.0
4,1003,1,1.0,NaN
6,1004,1,1.0,NaN
7,1004,2,NaN,NaN
8,1005,1,1.0,NaN
9,1005,2,1.0,NaN
10,1005,3,NaN,1.0
16,1011,1,NaN,1.0


### Narratives

In [73]:
dfs_eprime = pd.read_csv(os.path.join(Dir_dataset,'codes','fmri_passage_1001-1362.csv'))
df_story = pd.DataFrame({'raw':dfs_eprime['name'].unique()}).dropna()
df_story['raw'] = df_story['raw'].str.replace('.wav','')
df_story[['voice','story']] = df_story['raw'].str.split('_',n=1,expand=True)
df_story.groupby('story').count()

,raw,voice
story,,
AgedMother,2,2
Aqua,2,2
BiggestWastes,2,2
Bradford,1,1
BradfordBull,1,1
BulliesPronouns,1,1
BulliesRepeatedNames,1,1
Elvis,3,3
EmotionSentences_953,2,2


In [48]:
df_story.loc[df_story['story']=='Elvis']

,raw,voice,story
17,Jack_Elvis,Jack,Elvis
30,Brandi_Elvis,Brandi,Elvis
48,Jacob_Elvis,Jacob,Elvis
